In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install imagehash
import os
import json
import hashlib
from pathlib import Path
from PIL import Image
import imagehash
from tqdm import tqdm
from datasets import load_from_disk
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

DATASETS_ROOT = Path("/content/drive/MyDrive/RecoMind/Raw Data")
OUTPUT_PATH = Path("/content/drive/MyDrive/RecoMind/Processed Data/ar_caption_dataset.jsonl")

HASH_THRESHOLD = 3
SEED = 42

In [ ]:
translator = None

def init_translator():
    global translator
    if translator is None:
        print("Loading NLLB-200 in FP16 on CUDA...")

        tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-1.3B")
        model = AutoModelForSeq2SeqLM.from_pretrained(
            "facebook/nllb-200-1.3B",
            torch_dtype=torch.float16
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        translator = pipeline(
            "translation",
            model=model,
            tokenizer=tokenizer,
            src_lang="eng_Latn",
            tgt_lang="arb_Arab",
            device=0 if torch.cuda.is_available() else -1
        )


# def translate_en2ar(text):
#     if not text:
#         return ""
#     init_translator()
#     try:
#         return translator(text, max_length=128)[0]["translation_text"]
#     except Exception as e:
#         print(f" Translation error: {e}")
#         return ""

init_translator()

In [ ]:
def extract_samples(dataset_path):
    ds = load_from_disk(str(dataset_path))
    samples = []
    for row in tqdm(ds, desc=f"Extracting from {dataset_path.name}"):
        image = row.get("image")
        caption = row.get("text")
        if isinstance(image, Image.Image) and caption:
            hashname = hashlib.md5(image.tobytes()).hexdigest()
            img_path = dataset_path / f"{hashname}.jpg"
            image.convert("RGB").save(img_path, quality=95)
            samples.append({"image": str(img_path), "text_en": caption})
    return samples

In [ ]:
# def deduplicate(samples):
#     seen = {}
#     unique = []
#     for s in tqdm(samples, desc="Deduplicating"):
#         try:
#             h = imagehash.average_hash(Image.open(s["image"]))
#         except Exception as e:
#             print(f" Image error: {e}")
#             continue
#         if any(h - k < HASH_THRESHOLD for k in seen):
#             continue
#         seen[h] = s
#         unique.append(s)
#     return unique

In [ ]:
all_samples = []
dataset_folders = [
    "clothes_desc",
    "h-and-m-fashion-caption"
]

print(" Starting sampling!")

for folder in dataset_folders:
    folder_path = DATASETS_ROOT / folder
    samples = extract_samples(folder_path)
    all_samples.extend(samples)

print(" Total raw samples:", len(all_samples))

# unique_samples = deduplicate(all_samples)
# print(" Unique samples after deduplication:", len(unique_samples))
unique_samples = all_samples

BATCH_SIZE = 16

texts_en = [s["text_en"] for s in unique_samples]
texts_ar = []

for i in tqdm(range(0, len(texts_en), BATCH_SIZE), desc="Translating in batches"):
    batch = texts_en[i:i+BATCH_SIZE]
    translated = translator(batch, max_length=128)
    texts_ar.extend([t["translation_text"] for t in translated])

for s, ar_text in zip(unique_samples, texts_ar):
    s["text"] = ar_text
    del s["text_en"]

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for entry in unique_samples:
        json_str = json.dumps(entry, ensure_ascii=False)
        f.write(json_str + "\n")

print(f" Saved {len(unique_samples)} Arabic-captioned image-text pairs to:\n {OUTPUT_PATH}")

Here to continue

In [ ]:
# import json

# all_samples = []
# dataset_folders = [
#     # "kream-product-blip-captions",
#     "clothes_desc",
#     "h-and-m-fashion-caption"
# ]

# print(" Starting sampling!")

# for folder in dataset_folders:
#     folder_path = DATASETS_ROOT / folder
#     samples = extract_samples(folder_path)
#     all_samples.extend(samples)

# print("Total raw samples:", len(all_samples))

# # unique_samples = deduplicate(all_samples)
# # print(" Unique samples after deduplication:", len(unique_samples))

# # INTERMEDIATE_PATH = OUTPUT_PATH.parent / "deduplicated_samples.jsonl"
# # with open(INTERMEDIATE_PATH, "w", encoding="utf-8") as f:
# #     for sample in unique_samples:
# #         json_str = json.dumps(sample, ensure_ascii=False)
# #         f.write(json_str + "\n")

# # print(f" Saved deduplicated samples to:\n {INTERMEDIATE_PATH}")

Step 2: Load Deduplicated Samples + Batch Translation

In [ ]:
import json
from tqdm import tqdm
import os

INTERMEDIATE_PATH = OUTPUT_PATH.parent / "all_samples_together.jsonl"
unique_samples = []
with open(INTERMEDIATE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        unique_samples.append(json.loads(line))

print(f"Loaded {len(unique_samples)} all samples for translation")

In [ ]:
init_translator()

In [ ]:
import time

def load_translated_samples(path):
    if not os.path.exists(path):
        return []

    translated = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            entry = json.loads(line)
            translated.append(entry)
    return translated

saved_samples = load_translated_samples(OUTPUT_PATH)

BATCH_SIZE = 8
SAVE_EVERY = 15

if saved_samples:
    print(f"Resuming from {len(saved_samples)} saved translations")
    translated_texts = [s.get("text", "") for s in saved_samples]
    unique_samples = saved_samples
else:
    print("No saved translations found, starting fresh")

texts_en = []
indices_to_translate = []
for idx, s in enumerate(unique_samples):
    if "text" in s and s["text"]:
        continue
    texts_en.append(s["text_en"])
    indices_to_translate.append(idx)

texts_ar = [""] * len(unique_samples)

start = time.time()

for i in tqdm(range(0, len(texts_en), BATCH_SIZE), desc="Translating in batches"):
    batch = texts_en[i:i + BATCH_SIZE]
    try:
        translated = translator(batch, max_length=256)
        for j, t in enumerate(translated):
            idx = indices_to_translate[i + j]
            texts_ar[idx] = t["translation_text"]
    except Exception as e:
        print(f" Batch translation error at batch {i}: {e}")
        for j in range(len(batch)):
            idx = indices_to_translate[i + j]
            texts_ar[idx] = ""

    for idx in indices_to_translate[i:i + BATCH_SIZE]:
        unique_samples[idx]["text"] = texts_ar[idx]
        unique_samples[idx].pop("text_en", None)

    if ((i // BATCH_SIZE) + 1) % SAVE_EVERY == 0:
        with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
            for entry in unique_samples:
                json_str = json.dumps(entry, ensure_ascii=False)
                f.write(json_str + "\n")
        print(f"Saved progress after batch {(i // BATCH_SIZE) + 1}")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for entry in unique_samples:
        json_str = json.dumps(entry, ensure_ascii=False)
        f.write(json_str + "\n")

print(f"Saved {len(unique_samples)} Arabic-captioned image-text pairs to:\n {OUTPUT_PATH}")
print(f"Total translation time: {round(time.time() - start, 2)} seconds")
